# Improved Baseline: NFL Draft Prediction

This notebook keeps the competition rules intact:

- Uses only `train.csv`, `test.csv`, and `sample_submission.csv`.
- Generates model-based predictions only, with no hand-labeling.
- Sets seeds for reproducibility.
- Saves a valid `Id,Drafted` submission file.

The tutorial baseline Random Forest was about `0.813` CV AUC. The tuned Gradient Boosting workflow below reached about `0.832` mean 5-fold CV AUC locally.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

MODEL_SEED = 2025
SPLIT_SEED = 99  # Use 2025 to recreate the known 0.8365 public score file.
N_SPLITS = 5
np.random.seed(MODEL_SEED)


## 1. Load Data

The path detection below works in this project (`data/input`) and in the original competition folder (`input`).


In [ ]:
path_candidates = [
    Path("data/input"),
    Path("input"),
    Path("/content/drive/MyDrive/competition/input"),
]

PATH = next((path for path in path_candidates if (path / "train.csv").exists()), None)
if PATH is None:
    raise FileNotFoundError("Could not find train.csv. Put the CSV files in data/input or input.")

train = pd.read_csv(PATH / "train.csv")
test = pd.read_csv(PATH / "test.csv")
sample_submission = pd.read_csv(PATH / "sample_submission.csv")

print(f"Using data folder: {PATH}")
print(f"train: {train.shape}, test: {test.shape}, sample_submission: {sample_submission.shape}")
train.head()


## 2. Feature Engineering

These features use only columns already present in the provided data.


In [ ]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def add_features(df):
    df = df.copy()

    df["BMI"] = safe_divide(df["Weight"], df["Height"] ** 2)
    df["Weight_per_Height"] = safe_divide(df["Weight"], df["Height"])
    df["Broad_per_Height"] = safe_divide(df["Broad_Jump"], df["Height"])
    df["Vertical_per_Height"] = safe_divide(df["Vertical_Jump"], df["Height"])
    df["Bench_per_Weight"] = safe_divide(df["Bench_Press_Reps"], df["Weight"])

    df["Speed_Score"] = safe_divide(df["Weight"], df["Sprint_40yd"] ** 4)
    df["Power_Speed"] = safe_divide(df["Weight"], df["Sprint_40yd"])
    df["Jump_Power"] = df["Weight"] * df["Broad_Jump"]
    df["Explosive_Index"] = df["Vertical_Jump"] + df["Broad_Jump"]
    df["Agility_Index"] = df["Agility_3cone"] + df["Shuttle"]
    df["Agility_per_Weight"] = safe_divide(df["Agility_Index"], df["Weight"])

    return df

train_fe = add_features(train)
test_fe = add_features(test)

X = train_fe.drop(columns=["Id", "Drafted"])
y = train_fe["Drafted"].astype(int)
X_test = test_fe.drop(columns=["Id"])

print(X.shape, X_test.shape)


## 3. Preprocessing And Models

Numeric values are median-imputed with missing indicators. Categorical columns, including `School`, are one-hot encoded with unseen test categories ignored safely.


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    categorical_cols = [
        col for col in X.columns
        if pd.api.types.is_object_dtype(X[col].dtype)
        or pd.api.types.is_string_dtype(X[col].dtype)
        or isinstance(X[col].dtype, pd.CategoricalDtype)
    ]
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("one_hot", make_one_hot_encoder()),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols),
    ])


def candidate_models(seed=MODEL_SEED):
    return {
        "gradient_boosting": GradientBoostingClassifier(
            n_estimators=250,
            learning_rate=0.035,
            max_depth=2,
            min_samples_leaf=12,
            subsample=0.85,
            random_state=seed,
        ),
        "random_forest_12": RandomForestClassifier(
            n_estimators=900,
            max_depth=12,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed,
            n_jobs=1,
        ),
        "random_forest_9": RandomForestClassifier(
            n_estimators=800,
            max_depth=9,
            min_samples_leaf=4,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed,
            n_jobs=1,
        ),
        "hist_gradient_boosting_2": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.04,
            l2_regularization=0.1,
            max_leaf_nodes=12,
            min_samples_leaf=15,
            random_state=seed,
        ),
        "logistic": LogisticRegression(
            C=0.35,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs",
            random_state=seed,
        ),
    }

models = candidate_models()
models


## 4. Cross-Validation

This evaluates each model with stratified 5-fold CV and averages test predictions across folds.


In [ ]:
def build_pipeline(model):
    return Pipeline([
        ("preprocessor", build_preprocessor(X)),
        ("model", model),
    ])


def cross_validate_and_predict(models, X, y, X_test):
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SPLIT_SEED)
    oof_predictions = {}
    test_predictions = {}
    score_rows = []

    for name, model in models.items():
        print(f"\n{name}")
        oof = np.zeros(len(X))
        test_fold_preds = []
        fold_scores = []

        for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y), start=1):
            estimator = build_pipeline(clone(model))
            estimator.fit(X.iloc[train_idx], y.iloc[train_idx])

            valid_pred = estimator.predict_proba(X.iloc[valid_idx])[:, 1]
            test_pred = estimator.predict_proba(X_test)[:, 1]

            oof[valid_idx] = valid_pred
            test_fold_preds.append(test_pred)

            auc = roc_auc_score(y.iloc[valid_idx], valid_pred)
            fold_scores.append(auc)
            score_rows.append({"model": name, "fold": fold, "auc": auc})
            print(f"fold {fold}: {auc:.5f}")

        oof_predictions[name] = oof
        test_predictions[name] = np.mean(test_fold_preds, axis=0)
        print(f"mean AUC: {np.mean(fold_scores):.5f}; OOF AUC: {roc_auc_score(y, oof):.5f}")

    scores = pd.DataFrame(score_rows)
    return oof_predictions, test_predictions, scores

oof_predictions, test_predictions, scores = cross_validate_and_predict(models, X, y, X_test)
scores.groupby("model")["auc"].agg(["mean", "std"]).sort_values("mean", ascending=False)


## 5. Submission

The known public leaderboard score for the original Gradient Boosting fold-averaged submission was **0.8365**. This notebook now defaults to the best next attempt: the same model with `MODEL_SEED = 2025` and `SPLIT_SEED = 99`, which scored better in local OOF validation.

To recreate the known `0.8365` file instead, change `SPLIT_SEED` back to `2025` near the top of the notebook.


In [ ]:
best_model_name = "gradient_boosting"

submission = sample_submission.copy()
submission["Drafted"] = np.clip(test_predictions[best_model_name], 0, 1)

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
submission_path = output_dir / "submission.csv"
submission.to_csv(submission_path, index=False)

next_try_path = output_dir / "submission_next_try_gb_split_99.csv"
submission.to_csv(next_try_path, index=False)

print(f"Saved: {submission_path}")
print(f"Saved next candidate copy: {next_try_path}")
print(submission.shape)
submission.head()


## 6. Format Check


In [ ]:
assert list(submission.columns) == ["Id", "Drafted"]
assert len(submission) == len(test)
assert submission["Drafted"].between(0, 1).all()
assert not submission.isna().any().any()

submission.tail()


## 7. Next Steps

In this notebook, we built a stronger baseline model end to end, from preprocessing and feature engineering to model training, validation, and submission creation.

From here, this workflow can be used as a foundation for improving the competition score further. Good next experiments include:

- Trying different encoding methods for categorical features.
- Testing alternative missing-value imputation strategies.
- Experimenting with additional model types.
- Tuning model hyperparameters more carefully.
- Creating new feature engineering ideas from player measurements, positions, and athletic test results.

The goal is to keep improving the model while following the competition rules: use only the provided data, avoid hand-labeling, and keep the workflow reproducible.
